# Grounding Rust graph gaps before implementation

This notebook tests three separate contracts: physical source declarations, invocation/body evidence transport, and build-bound bounded panic reasoning. New candidate rows are experimental evidence, not a modified production graph. Complete syntax correspondence cannot establish complete panic freedom.

Inputs: the independently parsed 689-file Rust corpus and the newly rebuilt v11 spur-notebook graph. All source/graph identities and native compiler arguments are recorded. SOLVE checks finite identity and claim-promotion invariants; native Rust controls replay bounded witnesses.

In [3]:
from pathlib import Path
import json, hashlib, tarfile, gzip, io, os, subprocess, shutil, collections, time
import duckdb, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
ROOT = Path("/Users/kevintruong/project/SpurCode")
TARGET = Path("/Users/kevintruong/project/spur-notebook")
OLD = ROOT / "docs/research/artifacts/2026-11-02-rust-graph-coverage"
PREV = ROOT / "docs/research/artifacts/2026-11-02-spur-notebook-new-build-check"
OUT = ROOT / "docs/research/artifacts/2026-11-02-rust-gap-closure-grounding"
OUT.mkdir(parents=True, exist_ok=True)
for d in ["fixtures","captures","solver","tables","plots"]: (OUT/d).mkdir(exist_ok=True)
def sha(b): return hashlib.sha256(b).hexdigest()
def save(name, data):
    p=OUT/name; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(data,indent=2,ensure_ascii=False)+"\n")
    return data
def git_read(*args):
    r=subprocess.run(["git","-C",str(TARGET),*args],capture_output=True,text=True)
    assert r.returncode==0,r.stderr
    return r.stdout.strip()
current=str((TARGET/".spur/graph/CURRENT").resolve())
manifest=json.loads((PREV/"graph-manifest.json").read_text())
baseline=json.loads((OLD/"source-manifest.json").read_text())
print("source manifest shape", type(baseline).__name__, (list(baseline)[:8] if isinstance(baseline,dict) else baseline[:1]))
print("CURRENT",current,"graph manifest keys",list(manifest))
print("gaps sample",json.loads((PREV/"remaining-gaps.json").read_text())[:2])
print("archive members",end=" ")
with tarfile.open(OLD/"evidence.tar.gz","r:gz") as tf:
    names=tf.getnames()
    ast_name=next(n for n in names if n.endswith("rust-ast.json"))
    ast_payload=json.load(tf.extractfile(ast_name))
print(len(names), "AST", type(ast_payload).__name__, list(ast_payload)[:5] if isinstance(ast_payload,dict) else len(ast_payload))
project_before={"head":git_read("rev-parse","HEAD"),"status":git_read("status","--porcelain=v1"),"current":current}
save("project-before.json",project_before)
print("environment", {"duckdb":duckdb.__version__,"pyarrow":pa.__version__})


source manifest shape list [{'path': 'app_gallery/code-graph-app/backend/rust/metrics.rs', 'bytes': 566, 'lines': 19, 'sha256': '04b5cfe369f7e8a54384ca3ce4df01ee0785136106af024b752d6770b3c9e232', 'git_blob_oid': '7e59842702589db4cefbd5bf82b13a309dde5574'}]
CURRENT /Users/kevintruong/project/spur-notebook/.spur/graph/fcf40b9cd5785e09a0dbf3e45167f6e5f599348a44a1e184c7ff78d3e8e5cfec.parquet graph manifest keys ['graph_index_version', 'schema_version', 'manifest_version', 'graph_content_hash', 'indexed_commit_oid', 'extractor_version', 'complete', 'row_counts', 'macro_evidence', 'sidecar_complete', 'sidecar_row_counts', 'parquet_writer', 'edges_by_dst_present', 'temporal_shards']
gaps sample [{'file_path': 'crates/spurlab-telemetry/src/buffer.rs', 'kind': 'tuple_field', 'name': '<unnamed>', 'line_start': 66, 'start': 2505, 'end': 2518}, {'file_path': 'crates/spurlab-telemetry/src/qualification_tests.rs', 'kind': 'tuple_field', 'name': '<unnamed>', 'line_start': 375, 'start': 12061, 'end': 

1379 AST list 205726
environment {'duckdb': '1.5.5', 'pyarrow': '26.0.0'}


In [4]:
live_sources=[]
for x in baseline:
    b=(TARGET/x["path"]).read_bytes()
    assert sha(b)==x["sha256"], x["path"]
    live_sources.append(dict(x))
assert Path(current).name == manifest["graph_content_hash"]+".parquet"
assert json.loads((Path(current)/"manifest.json").read_text())==manifest
gaps=json.loads((PREV/"remaining-gaps.json").read_text())
gap_paths=sorted({r["file_path"] for r in gaps})
nodes=pq.read_table(PREV/"nodes.parquet").to_pylist()
declarations=pq.read_table(PREV/"declaration-coverage.parquet").to_pylist()
print("node keys",list(nodes[0]),"declaration keys",list(declarations[0]))
native_inputs=[]
with tarfile.open(OLD/"evidence.tar.gz","r:gz") as tf:
    for p in gap_paths:
        n=next(n for n in names if n.endswith("source-snapshot/"+p))
        b=tf.extractfile(n).read()
        assert b==(TARGET/p).read_bytes()
        native_inputs.append({"file_path":p,"source_oid":next(x["git_blob_oid"] for x in baseline if x["path"]==p),"source":b.decode()})
save("source-bindings.json",{"graph_content_hash":manifest["graph_content_hash"],"schema_version":manifest["schema_version"],"manifest_version":manifest["manifest_version"],"archive_sha256":sha((OLD/"evidence.tar.gz").read_bytes()),"sources":live_sources})
save("fixtures/structural-inputs.json",native_inputs)
print("verified sources",len(live_sources),"gap counts",dict(collections.Counter(g["kind"] for g in gaps)),"gap files",len(gap_paths))
inventory=(OLD/"inventory.evcxr.rs").read_text()
print(inventory[:4400])


node keys ['stable_symbol_id', 'node_id', 'file_path', 'byte_range_start', 'byte_range_end', 'line_start', 'line_end', 'entity_name', 'qualified_name', 'symbol_kind', 'anchor_hash', 'enclosing_scope'] declaration keys ['cfg_context', 'core_start', 'end', 'file_path', 'has_body', 'ident_end', 'ident_start', 'kind', 'line_end', 'line_start', 'name', 'row_type', 'start', 'test_context', 'ast_row_id', 'status', 'candidate_count', 'symbol_id']


verified sources 689 gap counts {'tuple_field': 446, 'extern_block': 3, 'foreign_function': 9} gap files 136
use syn::spanned::Spanned;
use syn::visit::{self, Visit};
use proc_macro2::Span;
use serde_json::{json, Value};

struct RustInventory {
    file: String, rows: Vec<Value>, test_context: bool, cfg_context: bool,
}
impl RustInventory {
    fn attr_flags(attrs: &[syn::Attribute]) -> (bool, bool) {
        let test = attrs.iter().any(|a| a.path().segments.last().map(|s|s.ident == "test").unwrap_or(false)) ||
            attrs.iter().any(|a| a.path().is_ident("cfg") && a.parse_args::<syn::Path>().map(|p|p.is_ident("test")).unwrap_or(false));
        let cfg = attrs.iter().any(|a| a.path().is_ident("cfg") || a.path().is_ident("cfg_attr"));
        (test, cfg)
    }
    fn push(&mut self, row_type: &str, kind: &str, name: String,
            span: Span, ident: Option<Span>, body: Option<bool>) {
        let range = span.byte_range();
        let idrange = ident.map(|s| s.byte_range());

In [3]:
:dep syn = {version = "=2.0.118", features = ["full", "visit"]}
:dep proc-macro2 = {version = "1", features = ["span-locations"]}
:dep serde_json = "1"

use syn::spanned::Spanned;
use syn::visit::{self, Visit};
use serde_json::{Value, json};
use proc_macro2::Span;

struct GapProbe { path:String, oid:String, source:String, rows:Vec<Value>, owners:Vec<Value> }
impl GapProbe {
    fn raw(&self, s:Span)->String { self.source[s.byte_range()].to_string() }
    fn loc(&self,s:Span)->Value { let r=s.byte_range(); json!({"start":r.start,"end":r.end}) }
    fn owner(&self)->Value { self.owners.last().cloned().unwrap_or(json!({"kind":"file","name":self.path,"start":0,"end":self.source.len()})) }
    fn tuple_fields(&mut self, f:&syn::Fields, owner:Value) {
        if let syn::Fields::Unnamed(u)=f {
            for (ordinal,field) in u.unnamed.iter().enumerate() {
                let r=field.span().byte_range();
                self.rows.push(json!({"file_path":self.path,"source_oid":self.oid,"kind":"tuple_field","name":ordinal.to_string(),"ordinal":ordinal,
                  "start":r.start,"end":r.end,"owner":owner,"raw_type":self.raw(field.ty.span()),
                  "type_range":self.loc(field.ty.span()),"visibility":self.raw(field.vis.span()),
                  "attributes":field.attrs.iter().map(|a|self.raw(a.span())).collect::<Vec<_>>(),"has_body":false}));
            }
        }
    }
}
impl<'ast> Visit<'ast> for GapProbe {
    fn visit_item(&mut self,item:&'ast syn::Item) {
        let named=match item {
            syn::Item::Fn(x)=>Some(("function",x.sig.ident.to_string())),
            syn::Item::Mod(x)=>Some(("module",x.ident.to_string())),
            syn::Item::Struct(x)=>Some(("struct",x.ident.to_string())),
            syn::Item::Enum(x)=>Some(("enum",x.ident.to_string())),
            _=>None
        };
        let pushed=named.is_some();
        if let Some((kind,name))=named {
            let r=item.span().byte_range();
            self.owners.push(json!({"kind":kind,"name":name,"start":r.start,"end":r.end}));
        }
        visit::visit_item(self,item);
        if pushed { self.owners.pop(); }
    }
    fn visit_item_struct(&mut self,x:&'ast syn::ItemStruct) {
        self.tuple_fields(&x.fields,self.owner());
        visit::visit_item_struct(self,x);
    }
    fn visit_variant(&mut self,x:&'ast syn::Variant) {
        let r=x.span().byte_range();
        let owner=json!({"kind":"enum_variant","name":x.ident.to_string(),"start":r.start,"end":r.end,"container":self.owner()});
        self.tuple_fields(&x.fields,owner);
        visit::visit_variant(self,x);
    }
    fn visit_item_foreign_mod(&mut self,x:&'ast syn::ItemForeignMod) {
        let r=x.span().byte_range();
        let block=json!({"kind":"extern_block","name":"<extern>","start":r.start,"end":r.end});
        let abi=x.abi.name.as_ref().map(|x|x.value()).unwrap_or("C".to_string());
        self.rows.push(json!({"file_path":self.path,"source_oid":self.oid,"kind":"extern_block","name":"<extern>",
          "start":r.start,"end":r.end,"owner":self.owner(),"abi":abi,"unsafe_block":x.unsafety.is_some(),
          "attributes":x.attrs.iter().map(|a|self.raw(a.span())).collect::<Vec<_>>(),"has_body":false}));
        for item in &x.items {
            if let syn::ForeignItem::Fn(f)=item {
                let fr=f.span().byte_range();
                self.rows.push(json!({"file_path":self.path,"source_oid":self.oid,"kind":"foreign_function","name":f.sig.ident.to_string(),
                  "start":fr.start,"end":fr.end,"owner":block,"abi":abi,"unsafe_block":x.unsafety.is_some(),
                  "signature":self.raw(f.sig.span()),"visibility":self.raw(f.vis.span()),
                  "attributes":f.attrs.iter().map(|a|self.raw(a.span())).collect::<Vec<_>>(),"has_body":false}));
            }
        }
        visit::visit_item_foreign_mod(self,x);
    }
}
let input_path="/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding/fixtures/structural-inputs.json";
let mut inputs:Vec<Value>=serde_json::from_slice(&std::fs::read(input_path)?)?;
let held_out=r#"
#[allow(dead_code)]
struct Pair(pub u8, #[cfg(any())] pub(crate) u8, u8);
enum E { Same(u8,u8), Named { x:u8 }, Unit }
mod nested { pub struct Pair(pub u8,pub u8); }
type NotAField = (u8,u8);
fn tuple_arg(_: (u8,u8)) { let _ = (1,2); }
unsafe extern "C" {
    #[link_name="alias_one"] fn imported(x:u8)->u8;
    static FOREIGN_COUNTER:u8;
}
mod ffi2 { unsafe extern "C-unwind" { fn imported(x:u8)->u8; } }
extern "C" fn defined_here(x:u8)->u8 { x }
"#;
inputs.push(json!({"file_path":"__fixture__/held-out.rs","source_oid":"fixture-not-git","source":held_out}));
let mut probe_rows:Vec<Value>=vec![];
for input in inputs {
    let source=input["source"].as_str().unwrap().to_string();
    let ast=syn::parse_file(&source)?;
    let mut p=GapProbe{path:input["file_path"].as_str().unwrap().into(),oid:input["source_oid"].as_str().unwrap().into(),source,rows:vec![],owners:vec![]};
    p.visit_file(&ast);
    probe_rows.extend(p.rows);
}
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding/captures/structural-syn-rows.json",serde_json::to_vec_pretty(&probe_rows)?)?;
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding/fixtures/structural-held-out.rs",held_out)?;
println!("{}",json!({"rows":probe_rows.len(),"real":probe_rows.iter().filter(|r|r["file_path"]!="__fixture__/held-out.rs").count(),"fixture":probe_rows.iter().filter(|r|r["file_path"]=="__fixture__/held-out.rs").count(),"parser":"syn 2.0.118"}));


The type of the variable spur was redefined, so was lost.


{"fixture":11,"parser":"syn 2.0.118","real":458,"rows":469}


In [5]:
probe_rows=json.loads((OUT/"captures/structural-syn-rows.json").read_text())
real=[r for r in probe_rows if not r["file_path"].startswith("__fixture__")]
fixture=[r for r in probe_rows if r["file_path"].startswith("__fixture__")]
key=lambda r:(r["file_path"],r["kind"],r["start"],r["end"])
assert set(map(key,real))==set(map(key,gaps))
assert len(real)==458==len(set(map(key,real)))
lookup=collections.defaultdict(list)
for d in declarations:
    lookup[(d["file_path"],d["kind"],d["name"],d["start"],d["end"])].append(d)
fms={r["path"]:r for r in pq.read_table(PREV/"file_manifests.parquet").to_pylist()}
def candidate_id(*parts):
    return "experimental:"+sha(json.dumps(parts,separators=(",",":"),ensure_ascii=False).encode())
block_ids={(r["file_path"],r["start"],r["end"]):candidate_id("extern_block",r["file_path"],r["source_oid"],r["start"],r["end"]) for r in real if r["kind"]=="extern_block"}
enriched=[]
for r in real:
    owner=r["owner"]
    if owner["kind"]=="extern_block":
        owner_id=block_ids[(r["file_path"],owner["start"],owner["end"])]
    elif owner["kind"]=="file":
        owner_id=fms[r["file_path"]]["stable_file_id"]
    else:
        ds=lookup[(r["file_path"],owner["kind"],owner["name"],owner["start"],owner["end"])]
        assert len(ds)==1 and ds[0]["status"]=="represented", (r,ds)
        owner_id=ds[0]["symbol_id"]
    ident=(candidate_id("tuple_field",owner_id,r["ordinal"]) if r["kind"]=="tuple_field" else
           block_ids[(r["file_path"],r["start"],r["end"])] if r["kind"]=="extern_block" else
           candidate_id("foreign_function",owner_id,r["name"]))
    enriched.append(dict(r,candidate_id=ident,owner_id=owner_id,physical_source_ordinal=r.get("ordinal"),compiled_ordinal=None,
        evidence_status="experimental_source_syntax",execution_status="no_executable_body",compiler_target="unknown"))
assert len({r["candidate_id"] for r in enriched})==458
# Equal field types must remain distinct; owner/ordinal is a source syntax identity, not compiled layout.
repeated_groups=collections.defaultdict(list)
for r in enriched:
    if r["kind"]=="tuple_field": repeated_groups[(r["owner_id"],r["raw_type"])].append(r)
same_type_groups=[g for g in repeated_groups.values() if len(g)>1]
assert all(len({r["candidate_id"] for r in g})==len(g) for g in same_type_groups)
save("captures/structural-candidates.json",enriched)
flat=[{"candidate_id":r["candidate_id"],"kind":r["kind"],"file_path":r["file_path"],"source_oid":r["source_oid"],"owner_id":r["owner_id"],
       "physical_ordinal":r.get("ordinal"),"start":r["start"],"end":r["end"],"name":r["name"],"raw_type":r.get("raw_type"),
       "abi":r.get("abi"),"has_body":r["has_body"],"execution_status":r["execution_status"]} for r in enriched]
table=pa.Table.from_pylist(flat)
pq.write_table(table,OUT/"tables/structural-candidates.parquet")
assert pq.read_table(OUT/"tables/structural-candidates.parquet").to_pylist()==flat
con=duckdb.connect()
relation=con.read_parquet(str(OUT/"tables/structural-candidates.parquet"))
print(relation.aggregate("kind,count(*) as candidate_rows,count(distinct candidate_id) as distinct_ids", "kind").df().to_string(index=False))
structural_results={"exact_original_gap_matches":len(real),"parent_bound_rows":len(enriched),"distinct_ids":len(set(r["candidate_id"] for r in enriched)),
 "same_owner_repeated_type_groups":len(same_type_groups),"fields_in_same_type_groups":sum(map(len,same_type_groups)),
 "held_out_rows":len(fixture),"held_out_kind_counts":dict(collections.Counter(r["kind"] for r in fixture)),
 "production_nodes_added":0,"parquet_roundtrip":True,"possible_syntax_correspondence_with_candidate_sidecar":"26507/26507",
 "compiled_ordinal":"unknown: cfg can remove fields and change physical-to-compiled positions"}
save("structural-results.json",structural_results)
print(json.dumps(structural_results,indent=2))
print("foreign declarations",[(r["file_path"],r["name"],r["abi"],r["attributes"]) for r in real if r["kind"]=="foreign_function"])


            kind  candidate_rows  distinct_ids
    extern_block               3             3
foreign_function               9             9
     tuple_field             446           446
{
  "exact_original_gap_matches": 458,
  "parent_bound_rows": 458,
  "distinct_ids": 458,
  "same_owner_repeated_type_groups": 3,
  "fields_in_same_type_groups": 6,
  "held_out_rows": 11,
  "held_out_kind_counts": {
    "tuple_field": 7,
    "extern_block": 2,
    "foreign_function": 2
  },
  "production_nodes_added": 0,
  "parquet_roundtrip": true,
  "possible_syntax_correspondence_with_candidate_sidecar": "26507/26507",
  "compiled_ordinal": "unknown: cfg can remove fields and change physical-to-compiled positions"
}
foreign declarations [('rest-table-gateway-ext/src/generation_probe_tests.rs', 'dlopen', 'C', []), ('rest-table-gateway-ext/src/generation_probe_tests.rs', 'dlsym', 'C', []), ('rest-table-gateway-ext/src/generation_probe_tests.rs', 'dlerror', 'C', []), ('rest-table-gateway-ext/src/nativ

In [8]:
import tempfile
SCRATCH=Path(tempfile.mkdtemp(prefix="spur-gap-grounding-",dir="/private/tmp"))
DETECTOR=Path("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
DEPS=DETECTOR.parent.parent/"deps"
env=dict(os.environ)
for k in list(env):
    if k.startswith(("CARGO_","RUSTFLAGS","RUSTDOCFLAGS")): env.pop(k,None)
env["RUSTUP_TOOLCHAIN"]="1.94.1-aarch64-apple-darwin"
env["TMPDIR"]="/private/tmp"
RUSTC=["rustup","run","1.94.1-aarch64-apple-darwin","rustc"]
def capture(name,argv,stdin=None,cwd=None,timeout=60):
    t=time.monotonic()
    p=subprocess.run([str(x) for x in argv],input=stdin,text=True,capture_output=True,cwd=cwd or SCRATCH,env=env,timeout=timeout)
    record={"argv":[str(x) for x in argv],"cwd":str(cwd or SCRATCH),"exit_code":p.returncode,
            "elapsed_seconds":time.monotonic()-t,"stdout":p.stdout,"stderr":p.stderr}
    save("captures/"+name+".json",record)
    return p
rv=capture("rustc-version",RUSTC+["-vV"])
assert rv.returncode==0
RUSTC_VERBOSE=rv.stdout
HOST=next(s.split(": ",1)[1] for s in rv.stdout.splitlines() if s.startswith("host:"))
print(RUSTC_VERBOSE)
print("available rlibs", {name:[p.name for p in DEPS.glob("lib"+name+"-*.rlib")] for name in ["spur_graph","serde_json"]})
driver=r'''
use std::io::Read;
fn main() -> Result<(),Box<dyn std::error::Error>> {
    let mut input=String::new(); std::io::stdin().read_to_string(&mut input)?;
    let v:serde_json::Value=serde_json::from_str(&input)?;
    let graph:spur_graph::GraphIndexArtifact=serde_json::from_value(v["graph"].clone())?;
    let owned:Vec<(String,String)>=v["sources"].as_array().ok_or("sources required")?.iter()
        .map(|s|(s["file_path"].as_str().unwrap().to_string(),s["text"].as_str().unwrap().to_string())).collect();
    let sources:Vec<_>=owned.iter().map(|(p,s)|spur_graph::panic_evidence::PanicEvidenceSource{file_path:p,bytes:s.as_bytes()}).collect();
    match spur_graph::panic_evidence::collect_rust_call_sites(&graph,&sources) {
        Ok(calls)=>{calls.validate(&graph,&sources)?; println!("{}",serde_json::json!({"status":"collected","calls":calls}));},
        Err(e)=>println!("{}",serde_json::json!({"status":"unknown","error":format!("{:#}",e)}))
    }
    Ok(())
}
'''
(OUT/"fixtures/call-collector-driver.rs").write_text(driver.strip()+"\n")
driver_src=SCRATCH/"call_collector.rs";driver_src.write_text(driver.strip()+"\n")
rg_lib=list(DEPS.glob("libspur_graph-*.rlib"))
sj_lib=list(DEPS.glob("libserde_json-*.rlib"))
assert len(rg_lib)==1 and len(sj_lib)>=1
sj_lib=sorted(sj_lib)[:1]  # explicit, recorded dependency selection; rustc checks crate compatibility
driver_bin=SCRATCH/"call_collector"
clang_resource=subprocess.run(["clang","--print-resource-dir"],capture_output=True,text=True,check=True).stdout.strip()
native_runtime=Path(clang_resource)/"lib/darwin"
assert (native_runtime/"libclang_rt.osx.a").exists(), str(native_runtime)
d=capture("compile-existing-collector-driver",RUSTC+[driver_src,"--edition=2021","--extern","spur_graph="+str(rg_lib[0]),"--extern","serde_json="+str(sj_lib[0]),"-L","dependency="+str(DEPS),"-L","native="+str(native_runtime),"-o",driver_bin])
assert d.returncode==0,d.stderr[-5000:]
save("collector-lineage.json",{"detector_sha256":sha(DETECTOR.read_bytes()),"driver_source_sha256":sha(driver_src.read_bytes()),
 "linked_graph_rlib":str(rg_lib[0]),"linked_graph_rlib_sha256":sha(rg_lib[0].read_bytes()),"linked_json_rlib":str(sj_lib[0]),
 "rustc_verbose":RUSTC_VERBOSE,"scope":"adapter calling existing public API; no production source changes"})
print("existing public collector adapter compiled", driver_bin)


rustc 1.94.1 (e408947bf 2026-03-25)
binary: rustc
commit-hash: e408947bfd200af42db322daf0fadfe7e26d3bd1
commit-date: 2026-03-25
host: aarch64-apple-darwin
release: 1.94.1
LLVM version: 21.1.8

available rlibs {'spur_graph': ['libspur_graph-e155f4e559a18b8e.rlib'], 'serde_json': ['libserde_json-9dd2d99b2ac7b492.rlib', 'libserde_json-375c0509d3ed4621.rlib']}


existing public collector adapter compiled /private/tmp/spur-gap-grounding-a1ji0pfe/call_collector


In [9]:
FIXTURES={
 "calls":{"src/lib.rs":"""fn leaf()->u8 { 1 }
pub fn caller(x:u8)->u8 {
    leaf(); leaf();
    let _deferred=|| leaf();
    let _future=async { leaf() };
    x.checked_add(1).unwrap()
}
""","src/zero.rs":"pub fn empty()->u8 { 1 }\n"},
 "cfg":{"src/lib.rs":"""#[cfg(feature="trip")]
pub fn selected(x:u8)->u8 { assert!(x!=1); x }
#[cfg(not(feature="trip"))]
pub fn selected(x:u8)->u8 { x }
"""},
 "macro_bounds":{"src/lib.rs":"""macro_rules! generated {
    () => { fn hidden(i:u8)->u8 { [7u8][i as usize] } }
}
generated!();
pub fn entry(i:u8)->u8 { hidden(i) }
"""},
 "overflow":{"src/lib.rs":"pub fn add_one(x:u8)->u8 { x+1 }\n"}
}
runs={}
for case,fs in FIXTURES.items():
    root=SCRATCH/case; root.mkdir()
    for path,text in fs.items():
        p=root/path;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(text)
        p2=OUT/"fixtures"/case/path;p2.parent.mkdir(parents=True,exist_ok=True);p2.write_text(text)
    req={"command":"snapshot","root":str(root),"source_paths":sorted(fs)}
    save("captures/"+case+"-snapshot-request.json",req)
    snap_p=capture(case+"-snapshot",[DETECTOR],json.dumps(req),cwd=root)
    assert snap_p.returncode==0,(case,snap_p.stderr)
    snap=json.loads(snap_p.stdout)
    call_req={"graph":snap["graph"],"sources":[{"file_path":p,"text":t} for p,t in sorted(fs.items())]}
    save("captures/"+case+"-calls-request.json",call_req)
    calls_p=capture(case+"-calls",[driver_bin],json.dumps(call_req),cwd=root)
    assert calls_p.returncode==0,(case,calls_p.stderr)
    calls=json.loads(calls_p.stdout)
    runs[case]={"snapshot":snap,"calls":calls,"sources":call_req["sources"]}
    save("captures/"+case+"-calls.json",calls)
    print(case,{"call_status":calls["status"],"bodies":len(calls.get("calls",{}).get("bodies",[])),
       "sites":len(calls.get("calls",{}).get("sites",[])),"reason":calls.get("error"),
       "source_symbols":[s["entity_name"] for s in snap["graph"]["symbols"] if not s["file_path"].startswith("external://")]})
assert runs["cfg"]["calls"]["status"]=="unknown" and "cfg" in runs["cfg"]["calls"]["error"]
actual_calls=runs["calls"]["calls"]["calls"]
assert len(actual_calls["sites"])==6
assert len({r["site_id"] for r in actual_calls["sites"]})==6
assert any(s["site_count"]==0 and s["file_path"]=="src/zero.rs" for s in actual_calls["sources"])
body_kind={b["body_id"]:b["kind"] for b in actual_calls["bodies"]}
print("exact invocation grains",[(s["call_text"],body_kind[s["body_id"]],s["site_id"][-12:]) for s in actual_calls["sites"]])


calls {'call_status': 'collected', 'bodies': 5, 'sites': 6, 'reason': None, 'source_symbols': ['leaf', 'caller', 'empty']}
cfg {'call_status': 'unknown', 'bodies': 0, 'sites': 0, 'reason': 'unevaluated cfg/cfg_attr prevents complete Rust call/body evidence', 'source_symbols': ['selected', 'selected']}
macro_bounds {'call_status': 'collected', 'bodies': 1, 'sites': 1, 'reason': None, 'source_symbols': ['generated', 'entry']}


overflow {'call_status': 'collected', 'bodies': 1, 'sites': 0, 'reason': None, 'source_symbols': ['add_one']}
exact invocation grains [('leaf()', 'function', '606909212d38'), ('leaf()', 'function', '271687dcce0f'), ('leaf()', 'closure', 'b1370e5fdddc'), ('leaf()', 'async_block', '8c3e574f252c'), ('x.checked_add(1)', 'function', '808426e6094b'), ('x.checked_add(1).unwrap()', 'function', '932cbeb97389')]


In [10]:
# Prototype the transport boundary with the existing production collector and validator.
validation_branch=r'''
    if let Some(candidate)=v.get("candidate") {
        let candidate:spur_graph::panic_evidence::RustCallSiteEvidenceSet=serde_json::from_value(candidate.clone())?;
        match candidate.validate(&graph,&sources) {
            Ok(())=>println!("{}",serde_json::json!({"status":"accepted"})),
            Err(e)=>println!("{}",serde_json::json!({"status":"rejected","error":format!("{:#}",e)}))
        }
        return Ok(());
    }
'''
driver_v2=driver.replace("    match spur_graph::panic_evidence::collect_rust_call_sites",validation_branch+"    match spur_graph::panic_evidence::collect_rust_call_sites")
driver_src.write_text(driver_v2.strip()+"\n")
(OUT/"fixtures/call-collector-driver.rs").write_text(driver_v2.strip()+"\n")
d=capture("compile-collector-validator-driver",RUSTC+[driver_src,"--edition=2021","--extern","spur_graph="+str(rg_lib[0]),"--extern","serde_json="+str(sj_lib[0]),"-L","dependency="+str(DEPS),"-L","native="+str(native_runtime),"-o",driver_bin])
assert d.returncode==0,d.stderr[-2000:]
lineage=json.loads((OUT/"collector-lineage.json").read_text())
lineage["driver_source_sha256"]=sha(driver_src.read_bytes())
lineage["driver_binary_sha256"]=sha(driver_bin.read_bytes())
save("collector-lineage.json",lineage)
for label,rows in [("call_sources",actual_calls["sources"]),("call_bodies",actual_calls["bodies"]),("call_sites",actual_calls["sites"])]:
    pq.write_table(pa.Table.from_pylist(rows),OUT/"tables"/(label+".parquet"))
transport=dict(actual_calls)
for label,key2 in [("call_sources","sources"),("call_bodies","bodies"),("call_sites","sites")]:
    transport[key2]=pq.read_table(OUT/"tables"/(label+".parquet")).to_pylist()
assert transport==actual_calls
base_req={"graph":runs["calls"]["snapshot"]["graph"],"sources":runs["calls"]["sources"]}
def check_candidate(name,candidate):
    request=dict(base_req,candidate=candidate)
    save("captures/validation-"+name+"-request.json",request)
    p=capture("validation-"+name,[driver_bin],json.dumps(request))
    assert p.returncode==0,p.stderr
    return json.loads(p.stdout)
checks={"roundtrip":check_candidate("roundtrip",transport)}
def clone(v): return json.loads(json.dumps(v))
bad=clone(transport); bad["sites"]=bad["sites"][1:]
checks["missing_site"]=check_candidate("missing-site",bad)
bad=clone(transport); bad["sites"].append(clone(bad["sites"][0]))
checks["duplicate_site"]=check_candidate("duplicate-site",bad)
bad=clone(transport); wrong=next(b["body_id"] for b in bad["bodies"] if b["kind"]=="closure")
bad["sites"][0]["body_id"]=wrong
checks["wrong_existing_body"]=check_candidate("wrong-existing-body",bad)
bad=clone(transport);bad["sites"][0]["source_oid"]="0"*40
checks["stale_source"]=check_candidate("stale-source",bad)
bad=clone(transport);bad["sources"]=[r for r in bad["sources"] if r["site_count"]!=0]
checks["missing_zero_site_source"]=check_candidate("missing-zero-site-source",bad)
assert checks["roundtrip"]["status"]=="accepted"
assert all(v["status"]=="rejected" for k,v in checks.items() if k!="roundtrip")
# Count how much source identity is irretrievably collapsed by a target-pair edge.
graph=runs["calls"]["snapshot"]["graph"]
syms={s["stable_symbol_id"]:s["entity_name"] for s in graph["symbols"]}
leaf_edges=[e for e in graph["edges"] if e.get("target_label")=="leaf" and e.get("relation")=="calls"]
print("edge encoding sample",graph["edges"][:1])
print("leaf call syntax sites",sum(s["call_text"]=="leaf()" for s in actual_calls["sites"]),"coarse leaf edges",len(leaf_edges))
call_results={"roundtrip":True,"sites":len(actual_calls["sites"]),"bodies":len(actual_calls["bodies"]),"source_rows":len(actual_calls["sources"]),
 "zero_site_sources":sum(s["site_count"]==0 for s in actual_calls["sources"]),
 "leaf_sites":sum(s["call_text"]=="leaf()" for s in actual_calls["sites"]),"leaf_body_kinds":dict(collections.Counter(body_kind[s["body_id"]] for s in actual_calls["sites"] if s["call_text"]=="leaf()")),
 "canonical_validation":checks,"schema_version":actual_calls["schema_version"],"extractor_version":actual_calls["extractor_version"],
 "production_persistence_changed":False}
save("call-transport-results.json",call_results)
print(json.dumps({k:v["status"] for k,v in checks.items()},indent=2))


edge encoding sample [{'source_stable_symbol_id': '0f4e270f5d072251', 'target_stable_symbol_id': None, 'target_label': 'checked_add', 'receiver_text': 'x', 'scope_text': 'u8', 'relation': 'calls', 'confidence': 'heuristic', 'confidence_score': 0.800000011920929, 'edge_kind': 'calls'}]
leaf call syntax sites 4 coarse leaf edges 1
{
  "roundtrip": "accepted",
  "missing_site": "rejected",
  "duplicate_site": "rejected",
  "wrong_existing_body": "rejected",
  "stale_source": "rejected",
  "missing_zero_site_source": "rejected"
}


In [11]:
# Native compiler controls run isolated programs, never project FFI or a workspace build.
native_results={}
harness=r'''
fn main() {
    std::panic::set_hook(Box::new(|_|{}));
    for x in 0u16..DOMAIN {
        let out=std::panic::catch_unwind(|| ENTRY(x as u8));
        match out {
            Ok(v)=>println!("{},{},{}",x,"returned",v),
            Err(e)=>{let p=e.downcast_ref::<String>().cloned().or_else(||e.downcast_ref::<&str>().map(|s|s.to_string())).unwrap_or("non-string".into());println!("{},{},{}",x,"panic",p)}
        }
    }
}
'''
configs=[
 ("overflow_on","overflow",True,False,256,"add_one"),
 ("overflow_off","overflow",False,False,256,"add_one"),
 ("macro_bounds","macro_bounds",True,False,4,"entry"),
 ("cfg_off","cfg",True,False,4,"selected"),
 ("cfg_on","cfg",True,True,4,"selected")
]
for name,case,overflow,trip,domain,entry in configs:
    root=SCRATCH/("native-"+name);root.mkdir()
    source=FIXTURES[case]["src/lib.rs"]+harness.replace("DOMAIN",str(domain)).replace("ENTRY",entry)
    src=root/"main.rs";src.write_text(source)
    (OUT/"fixtures"/(name+"-native.rs")).write_text(source)
    common=[src,"--edition=2021","-C","opt-level=0","-C","panic=unwind","-C","overflow-checks="+("on" if overflow else "off"),"-C","debug-assertions=off"]
    if trip: common+=["--cfg",'feature="trip"']
    binary=root/"control"
    c=capture(name+"-compile",RUSTC+common+["-o",binary])
    assert c.returncode==0,(name,c.stderr)
    m=capture(name+"-mir",RUSTC+common+["--emit=mir","-o",root/"control.mir"])
    assert m.returncode==0,(name,m.stderr)
    mir=(root/"control.mir").read_text();(OUT/"captures"/(name+".mir")).write_text(mir)
    p=capture(name+"-execute",[binary])
    assert p.returncode==0,p.stderr
    rows=[]
    for line in p.stdout.splitlines():
        x,status,payload=line.split(",",2)
        rows.append({"input":int(x),"outcome":status,"payload":payload})
    panics=[r["input"] for r in rows if r["outcome"]=="panic"]
    native_results[name]={"source_sha256":sha(src.read_bytes()),"binary_sha256":sha(binary.read_bytes()),"overflow_checks":overflow,
       "feature_trip":trip,"domain":[0,domain-1],"panics":panics,"rows":rows,"mir_assertions":sum("assert(" in l for l in mir.splitlines()),
       "mir_hidden_definition":"fn hidden(" in mir,"invocation_argv":json.loads((OUT/"captures"/(name+"-compile.json")).read_text())["argv"]}
    print(name,{"inputs":len(rows),"panic_inputs":panics,"MIR_asserts":native_results[name]["mir_assertions"],"MIR_hidden_fn":native_results[name]["mir_hidden_definition"]})
assert native_results["overflow_on"]["panics"]==[255]
assert native_results["overflow_off"]["panics"]==[]
assert native_results["macro_bounds"]["panics"]==[1,2,3]
assert native_results["cfg_on"]["panics"]==[1] and native_results["cfg_off"]["panics"]==[]
assert native_results["macro_bounds"]["mir_hidden_definition"]
# The same physical third tuple field becomes compiled positional field .1 under cfg(any()) = false.
tuple_source=(OUT/"fixtures/structural-held-out.rs").read_text()+'''
fn main(){let p=Pair(10,30); assert_eq!(p.0,10); assert_eq!(p.1,30); println!("physical_ordinal=2 compiled_ordinal=1 value={}",p.1);}
'''
tuple_src=SCRATCH/"tuple_cfg.rs";tuple_src.write_text(tuple_source)
(OUT/"fixtures/tuple-cfg-native.rs").write_text(tuple_source)
t=capture("tuple-cfg-compile",RUSTC+[tuple_src,"--edition=2021","-o",SCRATCH/"tuple_cfg"])
assert t.returncode==0,t.stderr
t=capture("tuple-cfg-execute",[SCRATCH/"tuple_cfg"])
assert t.returncode==0 and "compiled_ordinal=1 value=30" in t.stdout
save("native-results.json",native_results)
save("tuple-layout-control.json",{"stdout":t.stdout,"source_sha256":sha(tuple_src.read_bytes()),"source_ordinal":2,"compiled_ordinal":1,"cfg":"any() = false"})
print(t.stdout.strip())


overflow_on {'inputs': 256, 'panic_inputs': [255], 'MIR_asserts': 1, 'MIR_hidden_fn': False}


overflow_off {'inputs': 256, 'panic_inputs': [], 'MIR_asserts': 0, 'MIR_hidden_fn': False}


macro_bounds {'inputs': 4, 'panic_inputs': [1, 2, 3], 'MIR_asserts': 1, 'MIR_hidden_fn': True}


cfg_off {'inputs': 4, 'panic_inputs': [], 'MIR_asserts': 0, 'MIR_hidden_fn': False}


cfg_on {'inputs': 4, 'panic_inputs': [1], 'MIR_asserts': 0, 'MIR_hidden_fn': False}


physical_ordinal=2 compiled_ordinal=1 value=30


In [13]:
def build_declaration(graph,overflow=True,trip=False):
    argv=RUSTC+["--print","cfg","-C","overflow-checks="+("on" if overflow else "off"),"-C","debug-assertions=off"]
    if trip: argv+=["--cfg",'feature="trip"']
    cfg=capture("cfg-inventory-"+str(overflow)+"-"+str(trip),argv)
    assert cfg.returncode==0,cfg.stderr
    atoms=[]
    for s in cfg.stdout.splitlines():
        if "=" in s:
            k,v=s.split("=",1);atoms.append({"key":k,"value":json.loads(v)})
        else: atoms.append({"key":s,"value":None})
    atoms.sort(key=lambda a:(a["key"],a["value"] or ""))
    return {"schema_version":"panic-build-config-v1","graph_content_hash":graph["graph_content_hash"],"target":HOST,"edition":"2021","cfg":atoms,
        "overflow_checks":overflow,"debug_assertions":False,"panic_strategy":"unwind","rustc_verbose":RUSTC_VERBOSE,
        "codegen_flags":[],"features":["trip"] if trip else []}
semantic_runs={}
for name,case,overflow,trip in [
 ("overflow_on","overflow",True,False),("overflow_off","overflow",False,False),
 ("macro_bounds","macro_bounds",True,False),("cfg_on","cfg",True,True),("cfg_off","cfg",True,False),("calls","calls",True,False)]:
    build=build_declaration(runs[case]["snapshot"]["graph"],overflow,trip)
    req={"command":"analyze","graph":runs[case]["snapshot"]["graph"],"sources":runs[case]["sources"],"build":build}
    save("captures/"+name+"-analyze-request.json",req)
    p=capture(name+"-analyze",[DETECTOR],json.dumps(req))
    assert p.returncode==0,(name,p.stderr)
    result=json.loads(p.stdout)
    semantic_runs[name]={"request":req,"response":result}
    f=result.get("facts",{})
    print(name,{"status":result["status"],"bodies":len(f.get("bodies",[])),"obligations":len(f.get("unknown_obligations",[])),
        "operation_kinds":[o["operation"]["kind"] for o in f.get("extraction",{}).get("operations",[])],
        "obligation":result.get("obligation")})
solve_cases=[
 ("overflow_on_full","overflow_on","add_one",0,255),
 ("overflow_on_guarded_domain","overflow_on","add_one",0,254),
 ("overflow_off_full","overflow_off","add_one",0,255),
 ("macro_hidden_call","macro_bounds","entry",0,3),
 ("calls_deferred_bodies","calls","caller",0,255)]
detector_reports={}
for name,sem_name,entry,lo,hi in solve_cases:
    collected=semantic_runs[sem_name]
    facts=collected["response"]["facts"]
    syms={s["stable_symbol_id"]:s["entity_name"] for s in collected["request"]["graph"]["symbols"]}
    body=next(b["body_id"] for b in facts["bodies"] if syms.get(b["definition_stable_symbol_id"])==entry)
    req=dict(collected["request"])
    req["command"]="solve"; req["facts"]=facts
    typed_parameter=any(v.get("origin")=={"kind":"parameter","index":0} and v["context"]["location"]["body_id"]==body for v in facts["values"])
    req["domains"]=([{"body_id":body,"parameter_index":0,"domain":{"kind":"u8","minimum":lo,"maximum":hi}}] if typed_parameter else [])
    # Unknown parameter types retain obligations; do not invent a typed domain to force a query.
    req["limits"]={"entry_body_ids":[body],"expected_build_hash":facts["build_hash"],"max_call_depth":1,"max_loop_iterations":0,"max_states":64,"timeout_ms":10000,"persist":False}
    save("captures/"+name+"-solve-request.json",req)
    p=capture(name+"-solve",[DETECTOR],json.dumps(req))
    assert p.returncode==0,(name,p.stderr)
    result=json.loads(p.stdout);detector_reports[name]=result
    save("captures/"+name+"-report.json",result)
    qs=result["report"]["queries"]
    print(name,[(q["outcome"],q.get("response",{}).get("status"),q.get("candidate"),len(q.get("unresolved_obligations",[]))) for q in qs])
assert detector_reports["overflow_on_full"]["report"]["queries"][0]["outcome"]=="candidate_witness"
assert detector_reports["overflow_on_guarded_domain"]["report"]["queries"][0]["outcome"]=="no_counterexample_within_complete_scope"
assert detector_reports["overflow_off_full"]["report"]["queries"][0]["outcome"]=="no_counterexample_within_complete_scope"
assert detector_reports["macro_hidden_call"]["report"]["queries"][0]["outcome"]=="incomplete_semantics"
save("semantic-results.json",{k:{"status":v["response"]["status"],"obligation":v["response"].get("obligation"),
    "unknown_count":len(v["response"].get("facts",{}).get("unknown_obligations",[]))} for k,v in semantic_runs.items()})


overflow_on {'status': 'collected', 'bodies': 1, 'obligations': 0, 'operation_kinds': ['u8_add'], 'obligation': None}
overflow_off {'status': 'collected', 'bodies': 1, 'obligations': 0, 'operation_kinds': ['u8_add'], 'obligation': None}
macro_bounds {'status': 'collected', 'bodies': 1, 'obligations': 2, 'operation_kinds': [], 'obligation': None}
cfg_on {'status': 'unknown', 'bodies': 0, 'obligations': 0, 'operation_kinds': [], 'obligation': {'kind': 'cfg', 'detail': 'unevaluated cfg/cfg_attr prevents complete Rust call/body evidence'}}
cfg_off {'status': 'unknown', 'bodies': 0, 'obligations': 0, 'operation_kinds': [], 'obligation': {'kind': 'cfg', 'detail': 'unevaluated cfg/cfg_attr prevents complete Rust call/body evidence'}}
calls {'status': 'collected', 'bodies': 5, 'obligations': 12, 'operation_kinds': [], 'obligation': None}
overflow_on_full [('candidate_witness', 'sat', {'event_ids': ['1:55bc93d359872f2a580c66896b68a84a10cad10bda6ad4b117369e6916ba932c:operation:5'], 'inputs': {'i

{'overflow_on': {'status': 'collected',
  'obligation': None,
  'unknown_count': 0},
 'overflow_off': {'status': 'collected',
  'obligation': None,
  'unknown_count': 0},
 'macro_bounds': {'status': 'collected',
  'obligation': None,
  'unknown_count': 2},
 'cfg_on': {'status': 'unknown',
  'obligation': {'kind': 'cfg',
   'detail': 'unevaluated cfg/cfg_attr prevents complete Rust call/body evidence'},
  'unknown_count': 0},
 'cfg_off': {'status': 'unknown',
  'obligation': {'kind': 'cfg',
   'detail': 'unevaluated cfg/cfg_attr prevents complete Rust call/body evidence'},
  'unknown_count': 0},
 'calls': {'status': 'collected', 'obligation': None, 'unknown_count': 12}}

In [15]:
replay_req=json.loads((OUT/"captures/overflow_on_full-solve-request.json").read_text())
replay_req["limits"]["persist"]=True
p=capture("overflow-persisted-detector-solve",[DETECTOR],json.dumps(replay_req),cwd=ROOT)
assert p.returncode==0,p.stderr
detector_reports["overflow_on_full"]=json.loads(p.stdout)
save("captures/overflow-persisted-detector-report.json",detector_reports["overflow_on_full"])
replay_req["command"]="replay"
replay_req["report"]=detector_reports["overflow_on_full"]["report"]
replay_req["query_index"]=0
replay_req["expected_request_identity"]=replay_req["report"]["queries"][0]["request_identity"]
replay_req["expected_panic_payload"]="attempt to add with overflow"
replay_req["execution_timeout_ms"]=10000
save("captures/overflow-witness-replay-request.json",replay_req)
p=capture("overflow-witness-replay",[DETECTOR],json.dumps(replay_req),cwd=ROOT,timeout=30)
assert p.returncode==0,p.stderr
replay=json.loads(p.stdout)["replay"]
assert replay["runtime_reproduced"] is True and replay["actual_panic_payload"]=="attempt to add with overflow"
save("overflow-witness-replay.json",replay)
print({k:replay[k] for k in ["scope","outcome","runtime_reproduced","actual_panic_payload","build_hash","request_identity"]})
# Export the actual detector-produced generic constraints for independently persisted MCP execution.
generic_requests={}
for name,result in detector_reports.items():
    q=result["report"]["queries"][0]
    req=clone(q["request"]);req["persist"]=True;req["use_cache"]=False
    generic_requests[name]=req
save("solver/detector-generic-requests.json",generic_requests)
print("actual detector generic requests", {k:{"vars":len(v["vars"]),"constraints":len(v["constraints"])} for k,v in generic_requests.items()})


{'scope': 'standalone_fixture', 'outcome': 'reproduced', 'runtime_reproduced': True, 'actual_panic_payload': 'attempt to add with overflow', 'build_hash': 'sha256:7ca35a6d59b1ab74729e3132b492803bf16c6ccd078d913e044dd588dc330d8d', 'request_identity': 'sha256:9d30ef1f0253e6ab203f2c8d3d7c685f46d87022f17b0ebc4de0cb6ebd3730a1'}
actual detector generic requests {'overflow_on_full': {'vars': 2, 'constraints': 2}, 'overflow_on_guarded_domain': {'vars': 2, 'constraints': 2}, 'overflow_off_full': {'vars': 2, 'constraints': 2}, 'macro_hidden_call': {'vars': 0, 'constraints': 1}, 'calls_deferred_bodies': {'vars': 0, 'constraints': 1}}


In [17]:
def empty_facts():
    return {k:{} for k in ["relations","unique_constraints","foreign_keys","cardinality_constraints","value_ranges","conditional_requirements","aggregate_balances","consistency_relations","temporal_constraints"]}
def relation(fields,rows):
    return {"fields":fields,"rows":{str(i):{"active":True,"cells":{k:{"present":v is not None,"value":v} for k,v in r.items()}} for i,r in enumerate(rows)}}
def enum(values):return {"kind":"enum","values":sorted(set(values))}
def integer(lo=0,hi=100000):return {"kind":"integer","minimum":lo,"maximum":hi}
def rule(r,subject):return {"rule_id":"data_integrity."+r,"subjects":[subject]}
def request(f,rules):return {"family":"data_integrity","mode":"verify","facts":f,"rules":rules,"persist":True,"include_smt":True}
catalog_cases={}
sites=actual_calls["sites"];bodies=actual_calls["bodies"];sources=actual_calls["sources"]
site_ids=[s["site_id"] for s in sites];body_ids=[b["body_id"] for b in bodies];oids=[s["source_oid"] for s in sources]
fields={"site":enum(site_ids),"body":enum(body_ids),"oid":enum(oids+["STALE"])}  # 63 variables total; exact spans already canonical-source validated
rows=[{"site":s["site_id"],"body":s["body_id"],"oid":s["source_oid"]} for s in sites]
f=empty_facts()
f["relations"]={
 "sites":relation(fields,rows),
 "bodies":relation({"body":enum(body_ids)},[{"body":b} for b in body_ids]),
 "sources":relation({"oid":enum(oids+["STALE"])},[{"oid":s} for s in oids])}
f["unique_constraints"]={"site_identity":{"relation":"sites","fields":["site"]}}
f["foreign_keys"]={
 "site_body":{"child_relation":"sites","child_fields":["body"],"parent_relation":"bodies","parent_fields":["body"]},
 "site_source":{"child_relation":"sites","child_fields":["oid"],"parent_relation":"sources","parent_fields":["oid"]}}
f["cardinality_constraints"]={"site_count":{"relation":"sites","minimum":6,"maximum":6},"source_count":{"relation":"sources","minimum":2,"maximum":2}}
f["consistency_relations"]={"exact_source_owner":{"relation":"sites","fields":list(fields),"allowed":[[r[k] for k in fields] for r in rows]}}
rules=[rule("unique","site_identity"),rule("foreign_key","site_body"),rule("foreign_key","site_source"),rule("cardinality","site_count"),rule("cardinality","source_count"),rule("mutually_consistent","exact_source_owner")]
catalog_cases["post_actual_call_transport"]={"expect":"sat","request":request(f,rules)}
bad=clone(f);bad["relations"]["sites"]["rows"]["0"]["cells"]["body"]["value"]=next(b for b in body_ids if b!=rows[0]["body"])
catalog_cases["pre_wrong_owner_fk_only"]={"expect":"sat","request":request(bad,[rule("foreign_key","site_body")]),
 "interpretation":"A real but wrong body passes FK existence; exact owner correspondence is required."}
catalog_cases["post_wrong_owner_rejected"]={"expect":"unsat","request":request(bad,rules)}
for name,mutator in [
 ("duplicate_site",lambda f:f["relations"]["sites"]["rows"]["1"]["cells"]["site"].update(value=rows[0]["site"])),
 ("stale_source",lambda f:f["relations"]["sites"]["rows"]["0"]["cells"]["oid"].update(value="STALE")),
 ("missing_site",lambda f:f["relations"]["sites"]["rows"].pop("0")),
 ("missing_zero_source",lambda f:f["relations"]["sources"]["rows"].pop(str(next(i for i,s in enumerate(sources) if s["site_count"]==0))))
]:
    bad=clone(f);mutator(bad)
    catalog_cases["negative_"+name]={"expect":"unsat","request":request(bad,rules)}
real_pair=same_type_groups[0]
type_id=candidate_id("bad_type_only",real_pair[0]["owner_id"],real_pair[0]["raw_type"])
mapping=[{"candidate_id":r["candidate_id"],"ordinal":r["ordinal"],"owner_id":r["owner_id"],"source_oid":r["source_oid"],"raw_type":r["raw_type"]} for r in real_pair]
for phase,ids,expected in [("pre_type_only_collides",[type_id,type_id],"unsat"),("post_parent_ordinal_distinct",[r["candidate_id"] for r in real_pair],"sat")]:
    tf=empty_facts()
    tf["relations"]["fields"]=relation({"identity":enum(ids)},[{"identity":x} for x in ids])
    tf["unique_constraints"]["field_identity"]={"relation":"fields","fields":["identity"]}
    catalog_cases[phase]={"expect":expected,"request":request(tf,[rule("unique","field_identity")]),"source_projection":mapping}
ffi=[r for r in real if r["kind"] in ["extern_block","foreign_function"]]
ff=empty_facts()
ff["relations"]["ffi"]=relation({"kind":enum(["extern_block","foreign_function"]),"has_body":{"kind":"boolean"}},[{"kind":r["kind"],"has_body":r["has_body"]} for r in ffi])
ff["consistency_relations"]["no_fabricated_body"]={"relation":"ffi","fields":["kind","has_body"],"allowed":[["extern_block",False],["foreign_function",False]]}
catalog_cases["post_actual_ffi_body_status"]={"expect":"sat","request":request(ff,[rule("mutually_consistent","no_fabricated_body")])}
bad=clone(ff);bad["relations"]["ffi"]["rows"]["0"]["cells"]["has_body"]["value"]=True
catalog_cases["negative_ffi_fabricated_body"]={"expect":"unsat","request":request(bad,[rule("mutually_consistent","no_fabricated_body")])}
allowed=[
 ["syntax_only","source_syntax_coverage"],["cfg_unknown","unknown"],["incomplete_semantics","unknown"],
 ["candidate_witness","candidate"],["reproduced_witness","fixture_panic_reproduced"],
 ["complete_bounded_unsat","bounded_no_counterexample"],["fn_ptr_cast","address_flow"],["direct_call","call_observation"]]
states=sorted({x[0] for x in allowed});claims=sorted({x[1] for x in allowed}|{"full_panic_free"})
def claim_request(state,claim):
    facts={"components":{"evidence":{"selected":True,"attributes":{"state":state}},"report":{"selected":True,"attributes":{"claim":claim}}},
      "selection_groups":{},"version_orderings":{},
      "allowed_attribute_pairs":[{"left":{"component":"evidence","attribute":"state","values":states},"right":{"component":"report","attribute":"claim","values":claims},"allowed":allowed}]}
    return {"family":"configuration","mode":"verify","facts":facts,"rules":[{"rule_id":"configuration.attribute_allowed_pair","subjects":["evidence","report"]}],"persist":True,"include_smt":True}
for name,state,claim,expect in [
 ("post_incomplete_keeps_unknown","incomplete_semantics","unknown","sat"),
 ("negative_unsat_incomplete_promoted_safe","incomplete_semantics","bounded_no_counterexample","unsat"),
 ("negative_cfg_unknown_promoted_safe","cfg_unknown","bounded_no_counterexample","unsat"),
 ("post_bounded_unsat_claim","complete_bounded_unsat","bounded_no_counterexample","sat"),
 ("negative_syntax_coverage_panic_free","syntax_only","full_panic_free","unsat"),
 ("negative_address_flow_as_call","fn_ptr_cast","call_observation","unsat"),
 ("post_replayed_witness_claim","reproduced_witness","fixture_panic_reproduced","sat")]:
    catalog_cases[name]={"expect":expect,"request":claim_request(state,claim)}
save("solver/catalog-requests.json",catalog_cases)
# A manually normalized MIR/native bounds model supplements, never replaces, the incomplete source detector.
V=lambda name:{"kind":"var","name":name}
I=lambda n:{"kind":"int","value":n}
OP=lambda op,*args:{"kind":"op","op":op,"args":list(args)}
manual_requests={
 "mir_bounds_witness":{"vars":[{"type":"int_range","name":"i","min":0,"max":3}],"constraints":[{"id":"panic_bounds","expr":OP("ge",V("i"),I(1))}],"persist":True,"include_smt":True},
 "mir_bounds_guarded":{"vars":[{"type":"int_range","name":"i","min":0,"max":3}],"constraints":[{"id":"panic_bounds","expr":OP("ge",V("i"),I(1))},{"id":"branch_guard","expr":OP("lt",V("i"),I(1))}],"persist":True,"include_smt":True}
}
save("solver/manual-bounds-requests.json",manual_requests)
save("solver/normalization.json",{"catalog_cases":len(catalog_cases),"calls":"exact 6 production R3 sites / 5 bodies / 2 source bindings; domains copied as literal enums",
 "field_pair":mapping,"ffi_rows":len(ffi),"source_claim_contract":"caller-authored finite reporting policy, checked independently against observed detector/native results",
 "manual_bounds":"Explicitly normalized i >= 1 for array length 1 and i in 0..3; all 4 inputs replayed natively. Not an automatic translation of macro bodies.",
 "catalog_verify":"SAT = contract passes; UNSAT = invalid snapshot rejected","generic_query":"SAT = candidate; UNSAT only exact encoded scope. Incomplete report stays unknown."})
print("prepared",len(catalog_cases),"catalog requests and",len(generic_requests)+len(manual_requests),"generic requests")


prepared 18 catalog requests and 7 generic requests


In [2]:
from pathlib import Path
import json, hashlib, collections, subprocess, tempfile, os
ROOT=Path("/Users/kevintruong/project/SpurCode")
TARGET=Path("/Users/kevintruong/project/spur-notebook")
OUT=ROOT/"docs/research/artifacts/2026-11-02-rust-gap-closure-grounding"
def read(p): return json.loads((OUT/p).read_text())
def save(p,x):
    t=OUT/p;t.parent.mkdir(parents=True,exist_ok=True);t.write_text(json.dumps(x,indent=2,ensure_ascii=False)+"\n")
def sha(b):return hashlib.sha256(b).hexdigest()
cat=read("solver/catalog-results.json")
gen=read("solver/generic-results.json")
reloads=read("solver/reloaded-receipts.json")
assert len(cat)==18 and len(gen)==7 and len(reloads)==26
expected={}
for name,r in cat.items():
    assert r["response"]["status"]==r["expected"],name
    expected[r["response"]["solve_id"]]=r["response"]
for name,r in gen.items():expected[r["solve_id"]]=r
production=read("captures/overflow-persisted-detector-report.json")["report"]["queries"][0]["response"]
expected[production["solve_id"]]=production
for r in reloads:
    p=r["reloaded"];e=expected[r["solve_id"]]
    assert p["solve_id"]==r["solve_id"] and p["status"]==e["status"],r["name"]
    assert p.get("model")==e.get("model"),r["name"]
native=read("native-results.json")
replay=read("overflow-witness-replay.json")
assert replay["runtime_reproduced"] and replay["actual_panic_payload"]=="attempt to add with overflow"
assert gen["overflow_on_full"]["model"]["input_0"]==255
assert gen["mir_bounds_witness"]["model"]["i"] in native["macro_bounds"]["panics"]
assert gen["mir_bounds_guarded"]["status"]=="unsat"
for name in ["macro_hidden_call","calls_deferred_bodies"]:
    q=read("captures/"+name+"-report.json")["report"]["queries"][0]
    assert q["outcome"]=="incomplete_semantics" and q["response"]["status"]=="unsat"
rows=read("captures/structural-candidates.json")
gaps=json.loads((ROOT/"docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/remaining-gaps.json").read_text())
key=lambda r:(r["file_path"],r["kind"],r["start"],r["end"])
assert len(rows)==458 and len({r["candidate_id"] for r in rows})==458
assert set(map(key,rows))==set(map(key,gaps))
assert all(not r["has_body"] for r in rows)
bindings=read("source-bindings.json")
assert len(bindings["sources"])==689
for s in bindings["sources"]:assert sha((TARGET/s["path"]).read_bytes())==s["sha256"],s["path"]
summary={"catalog_checks":18,"catalog_pass_contracts":sum(r["response"]["status"]=="sat" for r in cat.values()),
 "catalog_rejected_contracts":sum(r["response"]["status"]=="unsat" for r in cat.values()),"generic_checks":7,
 "receipt_reload_count":26,"statuses":dict(collections.Counter(r["reloaded"]["status"] for r in reloads)),
 "native_inputs":sum(len(r["rows"]) for r in native.values()),"native_panic_inputs":sum(len(r["panics"]) for r in native.values()),
 "automatic_witness_replayed":True,"manual_bounds_witness":gen["mir_bounds_witness"]["model"],
 "complete_project_panic_detection":False}
save("solver/verification.json",summary)
# Confirm the independently audited two-pass ownership prediction with the real extractor.
src="enum E { A { x:u8 }, B { x:u8 }, T(u8,u8) }\npub fn use_e()->u8 { 1 }\n"
root=Path(tempfile.mkdtemp(prefix="gap-enum-ownership-",dir="/private/tmp"))
(root/"src").mkdir();(root/"src/lib.rs").write_text(src)
(OUT/"fixtures/enum-ownership.rs").write_text(src)
detector="/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence"
env=dict(os.environ);env["RUSTUP_TOOLCHAIN"]="1.94.1-aarch64-apple-darwin";env["TMPDIR"]="/private/tmp"
req={"command":"snapshot","root":str(root),"source_paths":["src/lib.rs"]}
p=subprocess.run([detector],input=json.dumps(req),text=True,capture_output=True,cwd=root,env=env,timeout=30)
assert p.returncode==0,p.stderr
save("captures/enum-ownership-snapshot.json",{"request":req,"exit_code":p.returncode,"stdout":p.stdout,"stderr":p.stderr})
og=json.loads(p.stdout)["graph"];physical=[s for s in og["symbols"] if s["file_path"]=="src/lib.rs"]
observed=[s["qualified_name"] for s in physical if s["entity_name"]=="x"]
save("enum-ownership-results.json",{"source_sha256":sha(src.encode()),"symbols":physical,"edges":og["edges"],
 "expected_named_owners":["E::A::x","E::B::x"],"observed_named_owners":observed,"production_extractor_changed":False})
print(json.dumps(summary,indent=2))
print("Actual enum owners:",observed)


{
  "catalog_checks": 18,
  "catalog_pass_contracts": 7,
  "catalog_rejected_contracts": 11,
  "generic_checks": 7,
  "receipt_reload_count": 26,
  "statuses": {
    "sat": 10,
    "unsat": 16
  },
  "native_inputs": 524,
  "native_panic_inputs": 5,
  "automatic_witness_replayed": true,
  "manual_bounds_witness": {
    "i": 1
  },
  "complete_project_panic_detection": false
}
Actual enum owners: ['E::x', 'E::x']


In [5]:
from pathlib import Path
import json,html,hashlib,collections,subprocess,gzip
import matplotlib.pyplot as plt
from IPython.display import HTML,display
ROOT=Path("/Users/kevintruong/project/SpurCode")
OUT=ROOT/"docs/research/artifacts/2026-11-02-rust-gap-closure-grounding"
TARGET=Path("/Users/kevintruong/project/spur-notebook")
load=lambda p:json.loads((OUT/p).read_text())
structural=load("structural-results.json");calls=load("call-transport-results.json")
native=load("native-results.json");solver=load("solver/verification.json")
records=load("captures/structural-candidates.json")
fig,axes=plt.subplots(1,3,figsize=(12,3.6))
fig.patch.set_facecolor("#f8fafc")
axes[0].barh(["Tuple fields","Foreign functions","Extern blocks"],[446,9,3],color=["#b45309","#d97706","#fbbf24"])
for y,v in enumerate([446,9,3]):axes[0].text(v+5,y,str(v),va="center",fontsize=11)
axes[0].set_xlim(0,510);axes[0].invert_yaxis();axes[0].set_title("458 source-bound candidate records")
axes[1].bar(["Exact leaf sites","Summary edges"],[4,1],color=["#2563eb","#94a3b8"])
axes[1].set_ylim(0,5);axes[1].set_title("Different invocation grains")
axes[1].text(0,4.1,"4",ha="center");axes[1].text(1,1.1,"1",ha="center")
axes[2].bar(["Checks on","Checks off"],[1,0],color=["#dc2626","#16a34a"])
axes[2].set_ylim(0,1.4);axes[2].set_title("Overflow panic inputs / 256")
axes[2].text(0,1.05,"x = 255",ha="center");axes[2].text(1,.06,"0",ha="center")
for ax in axes:
    ax.spines[["top","right"]].set_visible(False);ax.grid(axis="y",alpha=.12);ax.set_axisbelow(True)
fig.tight_layout()
(OUT/"plots").mkdir(exist_ok=True)
fig.savefig(OUT/"plots/grounding.png",dpi=170)
fig.savefig(OUT/"plots/grounding.svg")
plt.close(fig)
p=OUT/"plots/grounding.svg";p.write_text("\n".join(line.rstrip() for line in p.read_text().splitlines())+"\n")
def esc(x):return html.escape(str(x))
body_rows="".join("<tr><td>"+esc(r["kind"])+"</td><td>"+esc(r["file_path"])+"</td><td>"+esc(r.get("ordinal",""))+"</td><td><code>"+esc(r["owner_id"])+"</code></td><td>"+esc(str([r["start"],r["end"]]))+"</td></tr>" for r in records)
native_rows="".join("<tr><td>"+esc(k)+"</td><td>"+str(len(v["rows"]))+"</td><td>"+esc(v["panics"])+"</td></tr>" for k,v in native.items())
svg=(OUT/"plots/grounding.svg").read_text()
doc="""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Rust graph gap closure — experimental evidence</title>
<style>body{max-width:1150px;margin:36px auto;padding:0 22px;font:16px/1.55 system-ui;color:#17243a;background:#f8fafc}
h1{font-size:32px;line-height:1.2}h2{margin-top:32px}table{border-collapse:collapse;width:100%;background:white;font-size:14px}th,td{text-align:left;padding:8px;border-bottom:1px solid #dce3ec;vertical-align:top}th{position:sticky;top:0;background:#e8eef7}code{font-size:12px;overflow-wrap:anywhere}.card{padding:18px;background:white;border:1px solid #dce3ec;border-radius:10px;margin:16px 0}
.note{color:#475569}svg{max-width:100%;height:auto}.scroll{max-height:430px;overflow:auto}input{padding:10px;min-width:280px;border:1px solid #94a3b8;border-radius:6px}</style>
<h1>Rust graph gaps: source, invocation and panic evidence</h1>
<p>Source-bound experiments account for all <strong>458 missing declarations</strong>. Production still represents 26,049 / 26,507 inventory declarations; the new rows are an experimental sidecar.</p>
<div class="card">Repair ownership first: the current extractor gives both fields in <code>enum E { A { x:u8 }, B { x:u8 } }</code> the name <code>E::x</code>. Correct targets are <code>E::A::x</code> and <code>E::B::x</code>.</div>
""" +svg+"""
<h2>What the detector can establish</h2><p>The real source detector finds <code>x=255</code> for checked u8 overflow, and its own replay command reproduces the native panic. Macro-generated bounds checks and cfg-selected bodies retain explicit incomplete/unknown outcomes. Raw UNSAT with incomplete semantics is not a safety result.</p>
<table><thead><tr><th>Native configuration</th><th>Inputs</th><th>Panic inputs</th></tr></thead><tbody>"""+native_rows+"""</tbody></table>
<p class="note">524 native inputs; five panic cases across five configurations. A separate tuple control proves physical ordinal 2 can become compiled projection .1.</p>
<h2>Solver evidence</h2><p>18 catalog checks: 7 valid contracts pass, 11 invalid contracts reject. Seven generic queries plus the production witness receipt bring the total to <strong>26 persisted and reloaded receipts</strong>. All statuses/models matched. The manually normalized generated-bounds model gives i=1 and is checked against all four native inputs.</p>
<h2>Implementation order</h2><ol>
<li>Repair variant ownership before assigning IDs; capture tuple fields with physical ordinals.</li>
<li>Add source-backed foreign declarations and non-callable extern containers, preserving Rust lexical names and body absence.</li>
<li>Persist existing source/body/site evidence with availability, checksums, cache/delta/overlay handling and zero-site manifests.</li>
<li>Bind compiler observations to effective cfg/builds and expansion/trait/implicit-check provenance.</li>
<li>Reuse bounded semantic analysis, preserve uncertainty, and replay witnesses before reporting reproduction.</li></ol>
<p><a href="REPORT.md">Detailed findings and TDD acceptance plan</a> · <a href="solver/verification.json">Solver verification</a> · <a href="enum-ownership-results.json">Measured ownership defect</a></p>
<h2>Inspect all candidate declarations</h2><p><label>Filter by kind, path or owner: <input id="q" placeholder="tuple_field, native_registration, …"></label> <span id="n">458 rows</span></p>
<div class="scroll"><table id="gaps"><thead><tr><th>Kind</th><th>Source file</th><th>Physical ordinal</th><th>Owner identity</th><th>Byte range</th></tr></thead><tbody>"""+body_rows+"""</tbody></table></div>
<p class="note">All 689 Rust source hashes are fresh for the pinned v11 artifact. Compiler-selected membership, generated code, target correctness and complete project panic freedom remain distinct evidence requirements.</p>
<script>document.getElementById('q').addEventListener('input',e=>{let n=0;const q=e.target.value.toLowerCase();for(const r of document.querySelectorAll('#gaps tbody tr')){const show=r.textContent.toLowerCase().includes(q);r.hidden=!show;if(show)n++}document.getElementById('n').textContent=n+' rows'});</script></html>"""
(OUT/"report.html").write_text(doc)
report=(OUT/"REPORT.md").read_text().replace("../../../../../../2026-11-02-rust-gap-closure-grounding.ipynb","../../../../2026-11-02-rust-gap-closure-grounding.ipynb")
(OUT/"REPORT.md").write_text(report)
# Frozen source inputs are reproducible from the bound corpus; retain a compressed copy too.
p=OUT/"fixtures/structural-inputs.json"
if p.exists():
    with gzip.GzipFile(str(p)+".gz","wb",mtime=0) as gz:gz.write(p.read_bytes())
# Preserve target HEAD/status/CURRENT and the graph manifest, rather than assuming no changes.
def git(*args):
    p=subprocess.run(["git","-C",str(TARGET),*args],capture_output=True,text=True,check=True);return p.stdout.strip()
after={"head":git("rev-parse","HEAD"),"status":git("status","--porcelain=v1"),"current":str((TARGET/".spur/graph/CURRENT").resolve())}
before=load("project-before.json")
assert before==after,(before,after)
(OUT/"project-after.json").write_text(json.dumps(after,indent=2)+"\n")
manifest_before=json.loads((ROOT/"docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/graph-manifest.json").read_text())
manifest_after=json.loads((Path(after["current"])/"manifest.json").read_text())
changes={k:{"before":manifest_before.get(k),"after":manifest_after.get(k)} for k in set(manifest_before)|set(manifest_after) if manifest_before.get(k)!=manifest_after.get(k)}
assert set(changes)<= {"temporal_shards","sidecar_row_counts"}, list(changes)
# Compare source-backed node facts too: runtime node IDs may differ across rebuilds.
import pyarrow.parquet as pq
frozen=pq.read_table(ROOT/"docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/nodes.parquet").to_pylist()
live=[r for r in pq.read_table(Path(after["current"])/"nodes.parquet").to_pylist() if r["file_path"].endswith(".rs")]
keys=sorted(k for k in frozen[0] if k!="node_id")
canon=lambda rows:sorted(json.dumps({k:r[k] for k in keys},sort_keys=True) for r in rows)
assert canon(live)==canon(frozen),"live Rust symbol projection changed"
(OUT/"preservation-verification.json").write_text(json.dumps({"source_head_status_current_unchanged":True,"live_rust_symbol_rows":len(live),"rust_symbol_projection_matches":True,"manifest_changed_fields":sorted(changes),"scope":"temporal shard layout and embedding-sidecar counts changed during observation; frozen source/symbol evidence retained"},indent=2)+"\n")
print("Preserved source/symbol projection; observed ancillary manifest changes:",sorted(changes))
(OUT/"research-summary.json").write_text(json.dumps({"structural":structural,"calls":{k:calls[k] for k in ["sites","bodies","source_rows","zero_site_sources","roundtrip"]},"solver":solver,"target_source_and_selection_unchanged":True,"manifest_changed_fields":sorted(changes)},indent=2)+"\n")
display(HTML(doc))
print("Report, plots and preservation check saved; production implementation unchanged.")


Preserved source/symbol projection; observed ancillary manifest changes: ['sidecar_row_counts', 'temporal_shards']


Report, plots and preservation check saved; production implementation unchanged.


In [4]:
from pathlib import Path
import json
R=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts")
before=json.loads((R/"2026-11-02-spur-notebook-new-build-check/graph-manifest.json").read_text())
p=Path("/Users/kevintruong/project/spur-notebook/.spur/graph/CURRENT").resolve()/"manifest.json"
after=json.loads(p.read_text())
diff={k:{"before":before.get(k),"after":after.get(k)} for k in set(before)|set(after) if before.get(k)!=after.get(k)}
(R/"2026-11-02-rust-gap-closure-grounding/graph-manifest-after.json").write_text(json.dumps(after,indent=2)+"\n")
(R/"2026-11-02-rust-gap-closure-grounding/graph-manifest-changes.json").write_text(json.dumps(diff,indent=2)+"\n")
print(json.dumps(diff,indent=2)[:9000])


{
  "temporal_shards": {
    "before": [
      {
        "shard_idx": 0,
        "commit_time_min": 1779460918,
        "commit_time_max": 1780444972,
        "row_count_edges": 51615,
        "row_count_snapshots": 49457
      },
      {
        "shard_idx": 1,
        "commit_time_min": 1780444557,
        "commit_time_max": 1781258214,
        "row_count_edges": 85433,
        "row_count_snapshots": 84562
      },
      {
        "shard_idx": 2,
        "commit_time_min": 1781254026,
        "commit_time_max": 1781567068,
        "row_count_edges": 114058,
        "row_count_snapshots": 113604
      },
      {
        "shard_idx": 3,
        "commit_time_min": 1781572719,
        "commit_time_max": 1784691316,
        "row_count_edges": 51117,
        "row_count_snapshots": 48999
      },
      {
        "shard_idx": 4,
        "commit_time_min": 1784691759,
        "commit_time_max": 1785908412,
        "row_count_edges": 51879,
        "row_count_snapshots": 50654
      },
      {

In [6]:
from pathlib import Path
import runpy,json
base=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding")
module=runpy.run_path(str(base/"verify.py"))
verification=module["verify"](base,Path("/Users/kevintruong/project/spur-notebook"))
(base/"verification.json").write_text(json.dumps(verification,indent=2)+"\n")
print(json.dumps(verification,indent=2))


{
  "passed": true,
  "structural_rows": 458,
  "call_sites": 6,
  "call_bodies": 5,
  "saved_receipts": 26,
  "native_inputs": 524,
  "live_sources_checked": 689,
  "limits": "Checks saved evidence, canonical ranges/joins and recorded executions; does not re-run Z3/CGA or prove project panic freedom."
}


In [7]:
from pathlib import Path
import runpy,json
base=Path("/private/tmp/r8g-root-review-worktree/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding")
v=runpy.run_path(str(base/"verify.py"))["verify"](base,Path("/Users/kevintruong/project/spur-notebook"))
out=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-gap-closure-grounding")
(out/"isolated-verification.json").write_text(json.dumps(v,indent=2)+"\n")
print(json.dumps(v,indent=2))


{
  "passed": true,
  "structural_rows": 458,
  "call_sites": 6,
  "call_bodies": 5,
  "saved_receipts": 26,
  "native_inputs": 524,
  "live_sources_checked": 689,
  "limits": "Checks saved evidence, canonical ranges/joins and recorded executions; does not re-run Z3/CGA or prove project panic freedom."
}
